# ZIP to LLM Assets (Colab)

This notebook extracts a ZIP file (including nested folders) and converts supported files into LLM-friendly outputs.

## Output policy
- PDF: Markdown (and extracted images when available)
- DOCX: Markdown + embedded image extraction
- PPTX: Markdown + extracted slide images
- XLSX: Single Markdown file per workbook (all sheets combined)
- MP4: Transcript Markdown only
- Unsupported files: skipped and logged in manifest

In [13]:
import sys

if 'google.colab' in sys.modules:
    print('Installing Colab system dependencies...')
    !apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless ffmpeg
    print('Installing Python dependencies...')
    %pip install -q opendataloader-pdf[hybrid] python-docx python-pptx openpyxl pandas tabulate openai-whisper
else:
    print('Not running in Colab. Install these packages in your environment:')
    print('opendataloader-pdf[hybrid] python-docx python-pptx openpyxl pandas tabulate openai-whisper')

Installing Colab system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installing Python dependencies...


In [14]:
import socket
import time

print('Starting opendataloader PDF hybrid backend...')
!nohup opendataloader-pdf-hybrid --port 5002 > server.log 2>&1 &

for _ in range(30):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', 5002)) == 0:
            print('PDF backend is ready on port 5002')
            break
    time.sleep(1)
else:
    print('Warning: PDF backend status is uncertain. Last log lines:')
    !tail -n 20 server.log

Starting opendataloader PDF hybrid backend...
PDF backend is ready on port 5002


In [15]:
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

ZIP_PATH = '/content/drive/MyDrive/temp/input_bundle.zip'
WORK_DIR = '/content/zip2md_work'
EXTRACT_DIR = os.path.join(WORK_DIR, 'extracted')
OUTPUT_DIR = '/content/drive/MyDrive/zip2md_output'
REPORT_DIR = os.path.join(OUTPUT_DIR, 'reports')

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print('ZIP_PATH   =', ZIP_PATH)
print('EXTRACT_DIR=', EXTRACT_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f'ZIP file not found: {ZIP_PATH}')

Mounted at /content/drive
ZIP_PATH   = /content/drive/MyDrive/temp/input_bundle.zip
EXTRACT_DIR= /content/zip2md_work/extracted
OUTPUT_DIR = /content/drive/MyDrive/zip2md_output


In [16]:
import zipfile
import shutil

if os.path.isdir(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print('Extraction complete.')

all_files = []
for root, _, files in os.walk(EXTRACT_DIR):
    for name in files:
        full_path = os.path.join(root, name)
        rel_path = os.path.relpath(full_path, EXTRACT_DIR)
        all_files.append((rel_path, full_path))

print(f'Total discovered files: {len(all_files)}')
for rel_path, _ in all_files:   # Show all discovered files
    print(' -', rel_path)

Extraction complete.
Total discovered files: 177
 - GIFT/Jason Ng Yong Sheng.pdf
 - GIFT/ENG_ProgrammeSchedule.pdf
 - GIFT/ENG_ProgrammeSchedule.xlsx
 - GIFT/ENG_ProgrammeSchedule.xlsx - Table 1.pdf
 - GIFT/Orientation/O1_5_2PrepareforYourAdventure-UnlockthePowerofProblemSolving,DigitalToolsCommunicationPart2.pdf
 - GIFT/Orientation/O1_3_1HarnessYourStrengthsPartI.pdf
 - GIFT/Orientation/O1_3_2HarnessYourStrengthsPartII.pdf
 - GIFT/Orientation/O1_4AssessmentBriefing.pdf
 - GIFT/Orientation/O1_5_1PrepareforYourAdventure-UnlockthePowerofProblemSolving,DigitalToolsCommunicationPart1.pdf
 - GIFT/Orientation/O1_6PrepareforYourAdventure-BuildHabitsforSuccess.pdf
 - GIFT/Orientation/O1_2Confidence,Resilience,GrowthMindset.pdf
 - GIFT/Orientation/O1_1ALeapofFaith.pdf
 - GIFT/Scenario 1/Notes.docx
 - GIFT/Scenario 1/S1_pitch_JasonNgYongSheng.mp4
 - GIFT/Scenario 1/S1_8GenAImini-series1-CreatingaDigitalPortfoliowithGenAI.pdf
 - GIFT/Scenario 1/Copy of Cenergi-2022-Sustainability-Report-1.pdf
 - 

In [17]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import opendataloader_pdf
import pandas as pd
from docx import Document
from pptx import Presentation
import whisper

SUPPORTED = {".pdf", ".docx", ".pptx", ".xlsx", ".mp4"}

# Toggle this to rerun only files that were unsupported in the previous run.
PROCESS_ONLY_UNSUPPORTED = False
PREVIOUS_MANIFEST_PATH = os.path.join(REPORT_DIR, "conversion_manifest.json")

source_files = all_files
if PROCESS_ONLY_UNSUPPORTED and os.path.isfile(PREVIOUS_MANIFEST_PATH):
    with open(PREVIOUS_MANIFEST_PATH, "r", encoding="utf-8") as file_obj:
        previous_manifest = json.load(file_obj)

    retry_paths = [
        item.get("source_relative_path")
        for item in previous_manifest.get("unsupported", [])
        if item.get("source_relative_path")
    ]

    source_files = []
    for rel_path in retry_paths:
        full_path = os.path.join(EXTRACT_DIR, rel_path)
        if os.path.isfile(full_path):
            source_files.append((rel_path, full_path))

    print(f"Retry mode enabled: {len(source_files)} previously unsupported files selected")
else:
    print(f"Full mode: {len(source_files)} discovered files selected")

# Mirror the full extracted folder tree in OUTPUT_DIR so folder structure is complete,
# even when running in retry mode where only a subset of files is processed.
discovered_dirs = sorted({Path(rel_path).parent for rel_path, _ in all_files})
mirrored_dir_count = 0
for rel_dir in discovered_dirs:
    if str(rel_dir) in ("", "."):
        continue
    os.makedirs(os.path.join(OUTPUT_DIR, str(rel_dir)), exist_ok=True)
    mirrored_dir_count += 1
print(f"Mirrored folder tree into output: {mirrored_dir_count} folders")

manifest = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "zip_path": ZIP_PATH,
    "output_dir": OUTPUT_DIR,
    "processing_mode": "unsupported_only" if PROCESS_ONLY_UNSUPPORTED else "full",
    "discovered_file_count": len(all_files),
    "selected_file_count": len(source_files),
    "mirrored_folder_count": mirrored_dir_count,
    "processed": [],
    "failed": [],
    "unsupported": [],
}

mp4_model = None

for rel_path, full_path in source_files:
    ext = Path(rel_path).suffix.lower()
    rel_parent = Path(rel_path).parent
    out_base_dir = os.path.join(OUTPUT_DIR, str(rel_parent))
    os.makedirs(out_base_dir, exist_ok=True)

    if ext not in SUPPORTED:
        manifest["unsupported"].append(
            {
                "source_relative_path": rel_path,
                "reason": f"unsupported extension: {ext or '<none>'}",
            }
        )
        continue

    try:
        stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(rel_path).stem)

        if ext == ".pdf":
            opendataloader_pdf.convert(
                input_path=[full_path],
                output_dir=out_base_dir,
                format="markdown",
                hybrid="docling-fast",
                hybrid_mode="full",
                hybrid_fallback=True,
                use_struct_tree=True,
                quiet=True,
            )
            generated_md = os.path.join(out_base_dir, f"{Path(full_path).stem}.md")
            target_md = os.path.join(out_base_dir, f"{stem}_pdf.md")
            if os.path.isfile(generated_md):
                os.replace(generated_md, target_md)

            manifest["processed"].append(
                {
                    "source_relative_path": rel_path,
                    "type": "pdf",
                    "markdown": os.path.relpath(target_md, OUTPUT_DIR),
                }
            )

        elif ext == ".docx":
            doc = Document(full_path)
            md_lines = [f"# {Path(rel_path).name}", ""]

            for paragraph in doc.paragraphs:
                text = paragraph.text.strip()
                if not text:
                    continue
                style_name = (paragraph.style.name or "").lower()
                if "heading 1" in style_name:
                    md_lines.append(f"# {text}")
                elif "heading 2" in style_name:
                    md_lines.append(f"## {text}")
                elif "heading 3" in style_name:
                    md_lines.append(f"### {text}")
                else:
                    md_lines.append(text)
                md_lines.append("")

            for table_idx, table in enumerate(doc.tables, start=1):
                md_lines.append(f"## Table {table_idx}")
                rows = []
                for row in table.rows:
                    rows.append([cell.text.strip() for cell in row.cells])
                if rows:
                    header = rows[0]
                    md_lines.append("| " + " | ".join(header) + " |")
                    md_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
                    for row in rows[1:]:
                        padded = row + [""] * (len(header) - len(row))
                        md_lines.append("| " + " | ".join(padded[: len(header)]) + " |")
                    md_lines.append("")

            image_dir = os.path.join(out_base_dir, f"{stem}_images")
            os.makedirs(image_dir, exist_ok=True)
            image_count = 0
            rel_prefix = "" if str(rel_parent) in ("", ".") else f"{rel_parent.as_posix()}/"
            for rel in doc.part.rels.values():
                if "image" in rel.target_ref:
                    image_count += 1
                    img_suffix = Path(rel.target_ref).suffix or ".png"
                    img_name = f"image_{image_count:03d}{img_suffix}"
                    img_path = os.path.join(image_dir, img_name)
                    with open(img_path, "wb") as file_obj:
                        file_obj.write(rel.target_part.blob)
                    md_lines.append(f"![{img_name}]({rel_prefix}{stem}_images/{img_name})")

            target_md = os.path.join(out_base_dir, f"{stem}_docx.md")
            with open(target_md, "w", encoding="utf-8") as file_obj:
                file_obj.write("\n".join(md_lines).strip() + "\n")

            manifest["processed"].append(
                {
                    "source_relative_path": rel_path,
                    "type": "docx",
                    "markdown": os.path.relpath(target_md, OUTPUT_DIR),
                    "image_count": image_count,
                }
            )

        elif ext == ".pptx":
            prs = Presentation(full_path)
            md_lines = [f"# {Path(rel_path).name}", ""]
            slide_image_dir = os.path.join(out_base_dir, f"{stem}_slide_images")
            os.makedirs(slide_image_dir, exist_ok=True)
            slide_image_count = 0
            rel_prefix = "" if str(rel_parent) in ("", ".") else f"{rel_parent.as_posix()}/"

            for slide_index, slide in enumerate(prs.slides, start=1):
                md_lines.append(f"## Slide {slide_index}")
                slide_text_chunks = []

                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text and shape.text.strip():
                        slide_text_chunks.append(shape.text.strip())
                    if getattr(shape, "shape_type", None) == 13 and hasattr(shape, "image"):
                        slide_image_count += 1
                        img_suffix = Path(shape.image.filename).suffix or ".png"
                        img_name = f"slide_{slide_index:03d}_{slide_image_count:03d}{img_suffix}"
                        img_path = os.path.join(slide_image_dir, img_name)
                        with open(img_path, "wb") as file_obj:
                            file_obj.write(shape.image.blob)
                        md_lines.append(f"![{img_name}]({rel_prefix}{stem}_slide_images/{img_name})")

                if slide.has_notes_slide and slide.notes_slide.notes_text_frame:
                    notes_text = slide.notes_slide.notes_text_frame.text.strip()
                    if notes_text:
                        slide_text_chunks.append(f"Notes: {notes_text}")

                if slide_text_chunks:
                    md_lines.extend(slide_text_chunks)
                else:
                    md_lines.append("_No text found on this slide_")
                md_lines.append("")

            target_md = os.path.join(out_base_dir, f"{stem}_pptx.md")
            with open(target_md, "w", encoding="utf-8") as file_obj:
                file_obj.write("\n".join(md_lines).strip() + "\n")

            manifest["processed"].append(
                {
                    "source_relative_path": rel_path,
                    "type": "pptx",
                    "markdown": os.path.relpath(target_md, OUTPUT_DIR),
                    "image_count": slide_image_count,
                    "slide_count": len(prs.slides),
                }
            )

        elif ext == ".xlsx":
            xls = pd.ExcelFile(full_path)
            md_lines = [f"# {Path(rel_path).name}", ""]
            for sheet_name in xls.sheet_names:
                df = pd.read_excel(full_path, sheet_name=sheet_name)
                md_lines.append(f"## Sheet: {sheet_name}")
                if df.empty:
                    md_lines.append("_Empty sheet_")
                    md_lines.append("")
                    continue
                if len(df) > 500:
                    md_lines.append(f"_Truncated to first 500 rows out of {len(df)}_")
                    df = df.head(500)
                df = df.fillna("")
                md_lines.append(df.to_markdown(index=False))
                md_lines.append("")

            target_md = os.path.join(out_base_dir, f"{stem}_xlsx.md")
            with open(target_md, "w", encoding="utf-8") as file_obj:
                file_obj.write("\n".join(md_lines).strip() + "\n")

            manifest["processed"].append(
                {
                    "source_relative_path": rel_path,
                    "type": "xlsx",
                    "markdown": os.path.relpath(target_md, OUTPUT_DIR),
                    "sheet_count": len(xls.sheet_names),
                }
            )

        elif ext == ".mp4":
            if mp4_model is None:
                print("Loading Whisper model (base)...")
                mp4_model = whisper.load_model("base")
            result = mp4_model.transcribe(full_path)
            target_md = os.path.join(out_base_dir, f"{stem}_mp4_transcript.md")
            with open(target_md, "w", encoding="utf-8") as file_obj:
                file_obj.write(f"# Transcript: {Path(rel_path).name}\n\n")
                if result.get("segments"):
                    for segment in result["segments"]:
                        start = segment.get("start", 0.0)
                        end = segment.get("end", 0.0)
                        text = (segment.get("text") or "").strip()
                        if text:
                            file_obj.write(f"[{start:0.2f}s - {end:0.2f}s] {text}\n\n")
                else:
                    file_obj.write((result.get("text") or "").strip() + "\n")

            manifest["processed"].append(
                {
                    "source_relative_path": rel_path,
                    "type": "mp4",
                    "markdown": os.path.relpath(target_md, OUTPUT_DIR),
                }
            )

    except Exception as err:
        manifest["failed"].append(
            {
                "source_relative_path": rel_path,
                "error": str(err),
            }
        )

manifest["finished_at"] = datetime.now(timezone.utc).isoformat()
manifest["counts"] = {
    "processed": len(manifest["processed"]),
    "failed": len(manifest["failed"]),
    "unsupported": len(manifest["unsupported"]),
}

manifest_path = os.path.join(REPORT_DIR, "conversion_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as file_obj:
    json.dump(manifest, file_obj, indent=2)

print("Run complete.")
print(json.dumps(manifest["counts"], indent=2))
print(
    f"Discovered files: {manifest['discovered_file_count']} | "
    f"Selected files: {manifest['selected_file_count']}"
)
print("Manifest:", manifest_path)

Full mode: 177 discovered files selected
Mirrored folder tree into output: 21 folders
Loading Whisper model (base)...
Run complete.
{
  "processed": 177,
  "failed": 0,
  "unsupported": 0
}
Discovered files: 177 | Selected files: 177
Manifest: /content/drive/MyDrive/zip2md_output/reports/conversion_manifest.json


In [18]:
summary_md = os.path.join(REPORT_DIR, "conversion_summary.md")
with open(summary_md, "w", encoding="utf-8") as file_obj:
    file_obj.write("# Conversion Summary\n\n")
    file_obj.write(f"- Mode: {manifest.get('processing_mode', 'unknown')}\n")
    file_obj.write(f"- Discovered files: {manifest.get('discovered_file_count', 'n/a')}\n")
    file_obj.write(f"- Selected files: {manifest.get('selected_file_count', 'n/a')}\n")
    file_obj.write(f"- Mirrored folders: {manifest.get('mirrored_folder_count', 'n/a')}\n")
    file_obj.write(f"- Processed: {manifest['counts']['processed']}\n")
    file_obj.write(f"- Failed: {manifest['counts']['failed']}\n")
    file_obj.write(f"- Unsupported: {manifest['counts']['unsupported']}\n\n")

    file_obj.write("## Unsupported Files\n\n")
    if manifest["unsupported"]:
        for item in manifest["unsupported"]:
            file_obj.write(
                f"- {item['source_relative_path']} ({item['reason']})\n"
            )
    else:
        file_obj.write("- None\n")

    file_obj.write("\n## Failed Files\n\n")
    if manifest["failed"]:
        for item in manifest["failed"]:
            file_obj.write(f"- {item['source_relative_path']}: {item['error']}\n")
    else:
        file_obj.write("- None\n")

print("Summary report:", summary_md)

Summary report: /content/drive/MyDrive/zip2md_output/reports/conversion_summary.md


In [19]:
print('Stopping PDF backend...')
!pkill -f opendataloader-pdf-hybrid
print('Backend stop command sent.')

Stopping PDF backend...
Backend stop command sent.
